# Segmentación y cuantificación de núcleos con `scikit-image` + `matplotlib`

El objetivo de este ejercicio es realizar un flujo de trabajo simple con pasos de preprocesamiento, segmentación, operaciones morfológicas y cuantificación utilizando modulos usuales de `scikit-image` y amigos en un jupyter notebook.
 
Incluye:
- preprocesamiento (mediana + corrección de fondo),
- umbralización (incluyendo `try_all_threshold`),
- etiquetado y separación opcional de objetos tocantes,
- cuantificación morfológica básica,
- controles interactivos con `ipywidgets`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from skimage import filters, morphology, measure, segmentation, feature, color, util, data
from skimage.restoration import rolling_ball
from scipy import ndimage as ndi

import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['image.cmap'] = 'gray'

In [ ]:
def cargar_imagen_2d(ruta='data/nuclei.nd2'):
    ruta = Path(ruta)
    if ruta.exists():
        from bffile import BioFile
        with BioFile(str(ruta)) as bf:
            imagen = np.asarray(bf.as_array())
        # Si la imagen tiene dimensiones extra (T, Z, C, etc.),
        # tomamos el primer plano hasta quedarnos en 2D.
        while imagen.ndim > 2:
            imagen = imagen[0]
        return util.img_as_float(imagen), f'Imagen cargada desde: {ruta}'
    
    # Fallback para que el cuaderno siga funcionando sin ND2
    return util.img_as_float(data.cells3d()[30, 1]), 'No se encontró data/nuclei.nd2. Usando imagen de ejemplo (cells3d).'

In [ ]:
imagen, origen = cargar_imagen_2d('data/nuclei.nd2')
print(origen)
print('Forma:', imagen.shape, '| dtype:', imagen.dtype)

plt.figure(figsize=(6, 6))
plt.imshow(imagen)
plt.title('Imagen cruda')
plt.axis('off')
plt.show()

## 1) Preprocesamiento interactivo

Ajustá el radio de mediana y de rolling-ball para ver cómo cambia la corrección de fondo.

In [ ]:
def preprocesar(imagen, radio_mediana=3, radio_fondo=40):
    suave = filters.median(imagen, morphology.disk(radio_mediana))
    fondo = rolling_ball(suave, radius=radio_fondo)
    corregida = suave - fondo
    corregida = np.clip(corregida, a_min=0, a_max=None)
    return suave, fondo, corregida

@interact(
    radio_mediana=widgets.IntSlider(value=3, min=1, max=15, step=1),
    radio_fondo=widgets.IntSlider(value=40, min=5, max=120, step=1),
)
def explorar_preprocesamiento(radio_mediana=3, radio_fondo=40):
    suave, fondo, corregida = preprocesar(imagen, radio_mediana, radio_fondo)

    fig, ax = plt.subplots(1, 4, figsize=(16, 4))
    ax[0].imshow(imagen); ax[0].set_title('Cruda'); ax[0].axis('off')
    ax[1].imshow(suave); ax[1].set_title(f'Mediana (r={radio_mediana})'); ax[1].axis('off')
    ax[2].imshow(fondo); ax[2].set_title(f'Fondo (rolling-ball r={radio_fondo})'); ax[2].axis('off')
    ax[3].imshow(corregida); ax[3].set_title('Corregida'); ax[3].axis('off')
    plt.tight_layout()
    plt.show()

## 2) Umbralización: comparar métodos con `try_all_threshold`

Esta celda usa `skimage.filters.try_all_threshold` sobre la imagen preprocesada.

In [ ]:
suave_ref, fondo_ref, corregida_ref = preprocesar(imagen, radio_mediana=3, radio_fondo=40)

fig, ax = filters.try_all_threshold(corregida_ref, figsize=(14, 10), verbose=False)
plt.show()

## 3) Segmentación interactiva completa

Podés elegir método de umbral, tamaño mínimo de objeto y activar separación por watershed para objetos tocantes.

In [ ]:
def obtener_umbral(img, metodo='otsu'):
    metodos = {
        'otsu': filters.threshold_otsu,
        'yen': filters.threshold_yen,
        'li': filters.threshold_li,
        'isodata': filters.threshold_isodata,
        'triangle': filters.threshold_triangle,
    }
    return metodos[metodo](img)

def segmentar(corregida, metodo='otsu', min_size=64, separar_tocantes=True, min_distancia_picos=7):
    t = obtener_umbral(corregida, metodo)
    binaria = corregida > t
    binaria = morphology.remove_small_objects(binaria, min_size=min_size)

    if separar_tocantes:
        distancia = ndi.distance_transform_edt(binaria)
        coords = feature.peak_local_max(
            distancia,
            labels=binaria,
            min_distance=min_distancia_picos,
            exclude_border=False
        )
        marcadores_mask = np.zeros_like(binaria, dtype=bool)
        if len(coords) > 0:
            marcadores_mask[tuple(coords.T)] = True
        marcadores = measure.label(marcadores_mask)
        etiquetas = segmentation.watershed(-distancia, marcadores, mask=binaria)
    else:
        etiquetas = measure.label(binaria)

    return t, binaria, etiquetas

@interact(
    radio_mediana=widgets.IntSlider(value=3, min=1, max=15, step=1),
    radio_fondo=widgets.IntSlider(value=40, min=5, max=120, step=1),
    metodo=widgets.Dropdown(options=['otsu', 'yen', 'li', 'isodata', 'triangle'], value='otsu'),
    min_size=widgets.IntSlider(value=64, min=0, max=500, step=8),
    separar_tocantes=widgets.Checkbox(value=True),
    min_distancia_picos=widgets.IntSlider(value=7, min=1, max=20, step=1),
)
def explorar_segmentacion(radio_mediana, radio_fondo, metodo, min_size, separar_tocantes, min_distancia_picos):
    _, _, corregida = preprocesar(imagen, radio_mediana, radio_fondo)
    t, binaria, etiquetas = segmentar(corregida, metodo, min_size, separar_tocantes, min_distancia_picos)

    n_obj = int(etiquetas.max())
    superpuesta = color.label2rgb(etiquetas, image=corregida, bg_label=0, alpha=0.35)

    fig, ax = plt.subplots(1, 4, figsize=(18, 4))
    ax[0].imshow(corregida); ax[0].set_title('Preprocesada'); ax[0].axis('off')
    ax[1].imshow(binaria); ax[1].set_title(f'Máscara ({metodo}, t={t:.3f})'); ax[1].axis('off')
    ax[2].imshow(etiquetas, cmap='nipy_spectral'); ax[2].set_title(f'Etiquetas (n={n_obj})'); ax[2].axis('off')
    ax[3].imshow(superpuesta); ax[3].set_title('Overlay etiquetas'); ax[3].axis('off')
    plt.tight_layout()
    plt.show()

## 4) Cuantificación (regionprops)

Ejemplo con una configuración fija para extraer métricas por objeto.

In [ ]:
_, _, corregida_final = preprocesar(imagen, radio_mediana=3, radio_fondo=40)
_, _, etiquetas_final = segmentar(corregida_final, metodo='otsu', min_size=64, separar_tocantes=True, min_distancia_picos=7)

props = measure.regionprops_table(
    etiquetas_final,
    intensity_image=corregida_final,
    properties=['label', 'area', 'eccentricity', 'solidity', 'mean_intensity']
)
df = pd.DataFrame(props)
df.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(df['area'], bins=25, color='tab:blue', alpha=0.8)
ax[0].set_title('Distribución de áreas')
ax[0].set_xlabel('Área (px)')
ax[0].set_ylabel('Frecuencia')

ax[1].scatter(df['area'], df['mean_intensity'], s=25, alpha=0.7, color='tab:purple')
ax[1].set_title('Área vs intensidad media')
ax[1].set_xlabel('Área (px)')
ax[1].set_ylabel('Intensidad media')

plt.tight_layout()
plt.show()

print(f'Objetos medidos: {len(df)}')